# V033: input-only, bounded controller-bank correction

Training labels fit a controller bank. Inference receives known inputs and the accepted model prediction, never query pressure. One fixed conservative recipe; original A gate unchanged.


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import csv,json,hashlib,time,platform,inspect
from collections import Counter
import numpy as np
from threadpoolctl import threadpool_limits

V033_RECIPE=dict(neighbors=5,passes=4,distance_quantile=.9,consensus_physical_bins=1.,maximum_move_physical_bins=1,bootstrap_seed=20263301,bootstrap_draws=1000,minimum_gain=.003,threads=1)

def c33_need(c,m):
    if not bool(c):raise ValueError(m)
def c33_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def c33_code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
def c33_json(p,v):
    with Path(p).open('x') as f:f.write(json.dumps(v,indent=2,allow_nan=False)+'\n')
def c33_save(p,a):
    with Path(p).open('xb') as f:np.save(f,a,allow_pickle=False)
def c33_load_helper(root):
    path=Path(root)/'research/quantized_controller_components_v032.ipynb';code=c33_code(path)
    c33_need(hashlib.sha256(code.encode()).hexdigest()=='62b546ce5220958c5eadc2d604ae23779932ef24bc0bd76f67cc9bea6b3043d1','V032 component changed')
    ns={};exec(compile(code,str(path),'exec'),ns);return ns

def c33_load(path,breath_ids,*,pressure):
    """Membership filter precedes conversion. Known-input load never accesses pressure."""
    bids=np.sort(np.asarray(breath_ids,dtype=np.int64));c33_need(len(np.unique(bids))==len(bids),'Duplicate groups')
    at={int(b):i for i,b in enumerate(bids)};n=len(bids);x=np.empty((n,80,3),dtype=np.float64);rc=np.empty((n,2),dtype=np.int16);rows=np.empty((n,80),dtype=np.int64);count=np.zeros(n,dtype=np.int16);y=np.empty((n,80),dtype=np.float64) if pressure else None
    with Path(path).open(newline='') as f:
        for r in csv.DictReader(f):
            i=at.get(int(r['breath_id']))
            if i is None:continue
            j=int(count[i]);c33_need(j<80,'Too many rows');setting=(int(r['R']),int(r['C']))
            if j==0:rc[i]=setting
            else:c33_need(tuple(rc[i])==setting,'Within-breath R/C changed')
            rows[i,j]=int(r['id']);x[i,j]=[float(r['u_in']),float(r['time_step']),float(r['u_out'])]
            if pressure:y[i,j]=float(r['pressure'])
            count[i]+=1
    c33_need(np.all(count==80) and np.isfinite(x).all() and np.isin(x[:,:,2],[0.,1.]).all(),'Input quality/coverage')
    c33_need(np.all(np.diff(x[:,:,1],axis=1)>=0) and np.all(np.diff(rows,axis=1)>0),'Input order')
    if pressure:c33_need(np.isfinite(y).all(),'Target nonfinite')
    return bids,x,rc,rows,y

def c33_fit_bank(x,y,rc,helper):
    c33_need(x.shape==(len(y),80,3) and y.shape==(len(x),80),'Training shapes')
    lattice=helper['q32_lattice'](y[x[:,:,2]==0]);h=lattice['half_width'];indices=[];coef=[];delta=[];records=[]
    for i in range(len(x)):
        u=x[i,:,0];p=y[i];phase=x[i,:,2];f=helper['q32_prefix'](u,p,phase)
        rec=dict(training_index=i,status=f['status'],bank_eligible=False,eligible_points=len(f['k']))
        if f['status']=='physical':
            d,A,B,c=f['coef'];k=f['k'];r=d*u[k-1]-A*p[k]+B*p[k-1]+c-u[k]
            joint=helper['q32_joint'](r,k,A,B,h,1e-4)
            rec.update(full_joint=joint['feasible'],decay=float(d),A=float(A),B=float(B),intercept=float(c),condition=f['condition'],prefix_points=f['cut'])
            if joint['feasible']:
                indices.append(i);coef.append(f['coef']);delta.append(joint['delta']);rec['bank_eligible']=True
        records.append(rec)
    c33_need(len(indices)>5,'No sufficient coherent controller bank')
    model=dict(indices=np.asarray(indices,dtype=np.int64),coef=np.asarray(coef,dtype=np.float64),certificates=np.asarray(delta,dtype=np.float64),lattice=lattice)
    model.update(c33_index(x[model['indices']],rc[model['indices']]))
    return model,records

def c33_key(rc,phase):return (int(rc[0]),int(rc[1]),np.asarray(phase,dtype=np.uint8).tobytes())
def c33_d2(q,p):
    return np.maximum(np.einsum('ij,ij->i',q,q)[:,None]+np.einsum('ij,ij->i',p,p)[None,:]-2*q@p.T,0.)/240.

def c33_index(bank_inputs,bank_rc):
    """Known-input index; confidence is controls-only bank leave-one-out distance."""
    c33_need(bank_inputs.shape[1:]==(80,3),'Bank input shape')
    mu=bank_inputs.mean(axis=(0,1));sd=bank_inputs.std(axis=(0,1));sd=np.where(sd>0,sd,1.)
    features=((bank_inputs-mu)/sd).reshape(len(bank_inputs),240);groups={}
    for i in range(len(bank_inputs)):groups.setdefault(c33_key(bank_rc[i],bank_inputs[i,:,2]),[]).append(i)
    groups={k:np.asarray(v,dtype=np.int64) for k,v in groups.items()};threshold={};loo=np.full(len(bank_inputs),np.nan)
    for key,ii in groups.items():
        if len(ii)<6:continue
        pp=features[ii]
        for start in range(0,len(ii),64):
            use=np.arange(start,min(start+64,len(ii)));d2=c33_d2(pp[use],pp);d2[np.arange(len(use)),use]=np.inf
            order=np.argsort(d2,axis=1,kind='stable')[:,:5];loo[ii[use]]=np.sqrt(np.take_along_axis(d2,order,axis=1)[:,-1])
        threshold[key]=float(np.quantile(loo[ii],.9,method='linear'))
    return dict(bank_inputs=bank_inputs.copy(),bank_rc=bank_rc.copy(),mean=mu,std=sd,features=features,groups=groups,threshold=threshold,loo_fifth=loo)

def c33_retrieve(inputs,rc,model):
    """No pressures or IDs; exact R/C and known phase mask, stable bank-order ties."""
    features=((inputs-model['mean'])/model['std']).reshape(len(inputs),240);groups={}
    for i in range(len(inputs)):groups.setdefault(c33_key(rc[i],inputs[i,:,2]),[]).append(i)
    nn=np.full((len(inputs),5),-1,dtype=np.int64);dist=np.full((len(inputs),5),np.inf);confident=np.zeros(len(inputs),dtype=bool)
    for key,qq in groups.items():
        if key not in model['threshold']:continue
        ii=model['groups'][key];qq=np.asarray(qq);pp=model['features'][ii]
        for start in range(0,len(qq),64):
            use=qq[start:start+64];d2=c33_d2(features[use],pp);order=np.argsort(d2,axis=1,kind='stable')[:,:5]
            nn[use]=ii[order];dist[use]=np.sqrt(np.take_along_axis(d2,order,axis=1));confident[use]=dist[use,-1]<=model['threshold'][key]
    return nn,dist,confident

def c33_project(inputs,baseline,coefficients,passes=4):
    """Jacobi passes; baseline predictions anchor clipped/phase boundaries each pass."""
    c33_need(inputs.shape[1:]==(80,3) and baseline.shape==(len(inputs),80) and coefficients.shape==(len(inputs),5,4),'Projection shapes')
    u=inputs[:,:,0];phase=inputs[:,:,2];valid=np.zeros_like(baseline,dtype=bool)
    valid[:,2:]=(phase[:,2:]==0)&(phase[:,1:-1]==0)&(u[:,2:]>1e-6)&(u[:,2:]<99.999999)&(u[:,1:-1]>1e-6)&(u[:,1:-1]<99.999999)
    state=np.broadcast_to(baseline[:,None,:],(len(inputs),5,80)).copy();d,A,B,c=[coefficients[:,:,j,None] for j in range(4)]
    c33_need(np.all(A>B) and np.all(B>0),'Unstable/nonphysical coefficient bank')
    for _ in range(passes):
        update=(d*u[:,None,1:-1]+B*state[:,:,:-1][:,:,1:]+c-u[:,None,2:])/A
        state[:,:,2:]=np.where(valid[:,None,2:],update,baseline[:,None,2:])
    return state,valid

def c33_one_bin(baseline,proposal,gate,grid,lattice):
    """Move at most one existing physical level; no jump across a missing bin."""
    c33_need(baseline.shape==proposal.shape==gate.shape,'Correction shapes')
    g=np.asarray(grid,dtype=np.float64);c33_need(np.all(np.diff(g)>0) and len(g)>1,'Grid order')
    codes=np.rint((g-lattice['origin'])/lattice['spacing']).astype(np.int64)
    c33_need(len(np.unique(codes))==len(g),'Aliased physical grid codes')
    indices=np.searchsorted(g,baseline).clip(0,len(g)-1)
    c33_need(np.all(g[indices][gate]==baseline[gate]),'Eligible baseline must be on fitted pressure grid')
    direction=np.sign(proposal-baseline).astype(np.int64);neighbor=np.clip(indices+direction,0,len(g)-1);neighbor_value=g[neighbor]
    adjacent=(abs(codes[neighbor]-codes[indices])==1);mid=.5*(baseline+neighbor_value)
    crossing=((direction>0)&(proposal>mid))|((direction<0)&(proposal<=mid))
    return np.where(gate&adjacent&crossing,neighbor_value,baseline)

def c33_predict(inputs,rc,baseline,model,grid):
    """Input-only prediction API. Ground-truth query pressure is not an argument."""
    inputs=np.asarray(inputs,dtype=np.float64);baseline=np.asarray(baseline,dtype=np.float64)
    c33_need(inputs.shape==(len(rc),80,3) and baseline.shape==(len(rc),80) and np.isfinite(inputs).all() and np.isfinite(baseline).all(),'Query shape/finiteness')
    nn,dist,confidence=c33_retrieve(inputs,rc,model);coef=model['coef'][np.maximum(nn,0)]
    curves,valid=c33_project(inputs,baseline,coef,4);proposal=np.median(curves,axis=1);iqr=np.quantile(curves,.75,axis=1)-np.quantile(curves,.25,axis=1)
    gate=valid&confidence[:,None]&(iqr<=model['lattice']['spacing'])
    prediction=c33_one_bin(baseline,proposal,gate,grid,model['lattice'])
    return prediction,dict(neighbors=nn,distances=dist,confident=confidence,proposal=proposal,iqr=iqr,gate=gate,valid=valid)

def c33_bootstrap(candidate,baseline,truth,phase,seed=20263301):
    mask=phase==0;delta=np.where(mask,abs(candidate-truth)-abs(baseline-truth),0.).sum(1);counts=mask.sum(1);rng=np.random.default_rng(seed);draws=[]
    for _ in range(1000):
        ii=rng.integers(0,len(counts),len(counts));draws.append(float(delta[ii].sum()/counts[ii].sum()))
    return np.quantile(draws,[.025,.975]).tolist(),np.asarray(draws)

def c33_preflight(root):
    checks=[]
    def need(c,m):c33_need(c,m);checks.append(m)
    helper=c33_load_helper(root);need('q32_joint' in helper,'Pinned pure notebook helper loaded')
    g=np.arange(-5.,16.);lat=dict(origin=-5.,spacing=1.)
    b=np.array([[0.,0.,0.,0.,0.,0.]]);p=np.array([[3.,-3.,.5,-.5,.51,-.49]]);gate=np.ones_like(b,dtype=bool)
    need(np.array_equal(c33_one_bin(b,p,gate,g,lat),[[1.,-1.,0.,-1.,1.,0.]]),'Single-bin cap and lower exact-midpoint ties')
    need(np.array_equal(c33_one_bin(b,p,np.zeros_like(gate),g,lat),b),'Fallback is exact baseline')
    gap=np.array([-3.,0.,1.,2.]);need(c33_one_bin(np.array([[0.]]),np.array([[-3.]]),np.ones((1,1),bool),gap,lat)[0,0]==0,'Missing physical bin never enlarges cap')
    n=8;t=np.arange(80)*.03;x=np.zeros((n,80,3));x[:,:,0]=10+np.arange(n)[:,None]*.1;x[:,:,1]=t;x[:,40:,2]=1;rc=np.tile([20,20],(n,1));beta=np.tile([.9,2.,1.,1.],(n,1));index=c33_index(x,rc);index.update(coef=beta,lattice=dict(origin=-5.,spacing=1.));base=np.full((1,80),5.)
    pred,diag=c33_predict(x[:1],rc[:1],base,index,g);need(pred.shape==(1,80) and np.isfinite(pred).all(),'Input-only prediction finite')
    need(list(inspect.signature(c33_predict).parameters)==['inputs','rc','baseline','model','grid'],'Query prediction API excludes pressure and IDs')
    need(np.all(abs(pred-base)<=1.),'At most one physical pressure step')
    need(np.array_equal(pred[:,[0,1]],base[:,[0,1]]) and np.array_equal(pred[:,40:],base[:,40:]),'First two and expiration rows unchanged')
    need(np.all(np.diag(c33_d2(index['features'],index['features']))<1e-12),'Distance self-consistency')
    raw=np.array([np.linalg.norm(index['features'][0]-a)/np.sqrt(240) for a in index['features']]);need(np.allclose(diag['distances'][0],np.sort(raw)[:5],atol=1e-7),'Distances independently match direct norm')
    xx=x[:1].copy();xx[:,:,2]=1;fallback,_=c33_predict(xx,rc[:1],base,index,g);need(np.array_equal(fallback,base),'Unseen phase group falls back')
    xx=x[:1].copy();xx[:,:,0]=10000;fallback,dd=c33_predict(xx,rc[:1],base,index,g);need(not dd['confident'][0] and np.array_equal(fallback,base),'Out-of-distribution input distance falls back')
    tied=x.copy();tied[:,:,0]=10;ti=c33_index(tied,rc);nn,_,_=c33_retrieve(tied[:1],rc[:1],ti);need(np.array_equal(nn[0],np.arange(5)),'Stable canonical ties; IDs absent')
    means=index['mean'].copy();c33_predict(x[:1]*1.+0,rc[:1],base,index,g);need(np.array_equal(index['mean'],means),'Query does not refit scaling')
    control=x[:1].copy();control[0,4,0]=100;curves,valid=c33_project(control,base,np.tile([.9,2.,1.,1.],(1,5,1)));need(not valid[0,4] and not valid[0,5] and np.all(curves[:,:,4:6]==5),'Clipping resets anchors')
    # Independent scalar Jacobi recursion, never an in-place forward recurrence.
    expected=np.tile(base[:,None,:],(1,5,1))
    for _ in range(4):
        old=expected.copy()
        for j in range(80):
            if valid[0,j]:expected[0,:,j]=(.9*control[0,j-1,0]+old[0,:,j-1]+1-control[0,j,0])/2
    need(np.array_equal(curves,expected),'Four parallel passes agree with independent scalar recurrence')
    bp,_=c33_predict(np.repeat(x[:1],2,0),np.repeat(rc[:1],2,0),np.repeat(base,2,0),index,g);need(np.array_equal(bp[0],bp[1]),'Breath independence')
    # Poison labels in a synthetic CSV: known-input loading must not convert them.
    import tempfile
    with tempfile.TemporaryDirectory() as td:
        path=Path(td)/'toy.csv'
        with path.open('w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=['id','breath_id','R','C','u_in','time_step','u_out','pressure']);w.writeheader()
            for i in range(80):w.writerow(dict(id=i,breath_id=1,R=20,C=20,u_in=1,time_step=i*.03,u_out=int(i>=40),pressure='POISON'))
        loaded=c33_load(path,[1],pressure=False);need(loaded[-1] is None,'Known-input parser does not touch poisoned query pressures')
    ci,draws=c33_bootstrap(np.ones((6,80)),np.zeros((6,80)),np.zeros((6,80)),np.zeros((6,80)));need(ci==[1.,1.] and np.all(draws==1.),'Whole-breath bootstrap sign and native scale')
    need(len(c33_index(x[:5],rc[:5])['threshold'])==0,'Too-small groups cannot estimate leave-one-out fifth distance')
    return dict(status='passed',check_count=len(checks),checks=checks,no_approved_data_access=True)

def run_controller_bank_v033(root,*,expected_protocol_sha256,expected_source_sha256):
    root=Path(root).resolve();proto=root/'research/controller_bank_protocol_v033.json';source=root/'research/controller_bank_components_v033.ipynb'
    c33_need(c33_sha(proto)==expected_protocol_sha256 and hashlib.sha256(c33_code(source).encode()).hexdigest()==expected_source_sha256,'Own protocol/source changed')
    protocol=json.loads(proto.read_text());c33_need(protocol['recipe']==V033_RECIPE,'Recipe mismatch')
    for rel,h in protocol['input_hashes'].items():c33_need(c33_sha(root/rel)==h,'Input changed: '+rel)
    o=root/'outputs/controller_bank_screen_v033';c33_need(not o.exists(),'Preserve previous run');o.mkdir()
    start=time.perf_counter();c33_json(o/'preflight.json',c33_preflight(root));split=np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz')
    fit_ids=split['A_fit'];val_ids=split['A_validation'];c33_need(len(fit_ids)==61114 and len(val_ids)==6791 and not np.intersect1d(fit_ids,val_ids).size,'Frozen split')
    print('Fitting input-only controller bank from original Afit',flush=True)
    with threadpool_limits(limits=1):
        bids,x,rc,rows,y=c33_load(root/'data/train.csv',fit_ids,pressure=True);helper=c33_load_helper(root);model,fit_records=c33_fit_bank(x,y,rc,helper)
        import pandas as pd
        pd.DataFrame(fit_records).to_csv(o/'training_bank_fit_records.csv',index=False)
        grid=np.unique(y[x[:,:,2]==0].astype(np.float32)).astype(np.float64);c33_need(np.array_equal(grid,np.load(root/'outputs/postprocessing_screen_v016/A_fit_pressure_codebook.npy')),'Fitted inspiration codebook mismatch')
        c33_json(o/'lattice.json',model['lattice'])
        for name in ['indices','coef','certificates','mean','std','loo_fifth']:c33_save(o/('bank_'+name+'.npy'),model[name])
        c33_save(o/'bank_breath_ids.npy',bids[model['indices']]);c33_save(o/'training_pressure_codebook.npy',grid)
        c33_json(o/'bank_index.json',dict(bank_size=len(model['indices']),physical_count=sum(r['status']=='physical' for r in fit_records),group_count=len(model['groups']),eligible_group_count=len(model['threshold']),groups=[dict(R=k[0],C=k[1],phase=list(k[2]),bank_indices=model['groups'][k].tolist(),fifth_distance_threshold=model['threshold'].get(k)) for k in model['groups']],scaling_scope='coherent Afit controller-bank known inputs only',full_eligible_equations_certified=True))
        print('Bank ready:',len(model['indices']),'controllers;',len(model['threshold']),'eligible input groups',flush=True)
        # Query labels are deliberately absent until every candidate prediction is frozen.
        vb,vx,vrc,vr,_=c33_load(root/'data/train.csv',val_ids,pressure=False);mask=vx[:,:,2]==0;ids=vr[mask]
        c33_need(np.array_equal(ids,np.load(root/'outputs/controlled_experiments_v013/A_row_ids.npy')) and mask.sum()==206098,'Accepted prediction row order')
        baseline=np.zeros((len(vb),80),dtype=np.float64);baseline[mask]=np.load(root/'outputs/postprocessing_screen_v016/weighted_median_midpoint_then_nearest_training_pressure_codebook_A_predictions.npy')
        prediction,diagnostic=c33_predict(vx,vrc,baseline,model,grid)
        c33_save(o/'A_predictions.npy',prediction[mask]);c33_save(o/'A_row_ids.npy',ids);c33_save(o/'A_breath_ids.npy',vb)
        for name,a in diagnostic.items():c33_save(o/('A_'+name+'.npy'),a)
        c33_json(o/'prediction_locked_before_labels.json',dict(created_utc=datetime.now(timezone.utc).isoformat(),prediction_sha256=c33_sha(o/'A_predictions.npy'),row_ids_sha256=c33_sha(o/'A_row_ids.npy'),source_code_sha256=expected_source_sha256,protocol_sha256=expected_protocol_sha256,query_pressure_absent=True,neural_training=0,neural_checkpoint_inferences=0))
        print('Candidate predictions locked; loading labels for scoring only',flush=True)
        tb,tx,trc,trows,truth=c33_load(root/'data/train.csv',val_ids,pressure=True);c33_need(np.array_equal(tb,vb) and np.array_equal(tx,vx) and np.array_equal(trows,vr),'Score alignment')
        truth=truth.astype(np.float32).astype(np.float64);be=abs(baseline-truth);pe=abs(prediction-truth);base_mae=float(be[mask].mean());mae=float(pe[mask].mean());gain=base_mae-mae
        c33_need(abs(base_mae-.18292209456473527)<1e-12,'Accepted baseline score mismatch')
        interval,draws=c33_bootstrap(prediction,baseline,truth,vx[:,:,2]);c33_save(o/'bootstrap_deltas.npy',draws);eligible=bool(gain>=.003 and interval[1]<0)
        rcrows=[]
        for setting in sorted(set(map(tuple,vrc))):
            use=np.all(vrc==setting,axis=1);m=mask[use];err=pe[use][m];old=be[use][m];rcrows.append(dict(R=int(setting[0]),C=int(setting[1]),scored_points=int(m.sum()),MAE=float(err.mean()),baseline_MAE=float(old.mean()),total_absolute_error=float(err.sum()),delta_total_absolute_error=float(err.sum()-old.sum()),changed_points=int(np.sum((prediction[use]!=baseline[use])&m))))
        pd.DataFrame(rcrows).to_csv(o/'RC_errors.csv',index=False)
        pos=[]
        for j in range(80):
            m=mask[:,j]
            if m.any():pos.append(dict(position=j,scored_points=int(m.sum()),MAE=float(pe[m,j].mean()),baseline_MAE=float(be[m,j].mean()),total_absolute_error=float(pe[m,j].sum())))
        pd.DataFrame(pos).to_csv(o/'position_errors.csv',index=False)
        changed=(prediction!=baseline)&mask
        result=dict(status='complete',completed_utc=datetime.now(timezone.utc).isoformat(),protocol_sha256=expected_protocol_sha256,source_code_sha256=expected_source_sha256,recipe=V033_RECIPE,reference_A_MAE=base_mae,MAE=mae,median_absolute_error=float(np.median(pe[mask])),gain=gain,bootstrap_95=interval,eligible=eligible,selected_for_confirmation='controller_bank_one_bin' if eligible else None,bank_size=len(model['indices']),validation_breaths=len(vb),validation_scored_points=int(mask.sum()),distance_confident_breaths=int(diagnostic['confident'].sum()),gate_points=int((diagnostic['gate']&mask).sum()),changed_points=int(changed.sum()),changed_breaths=int(changed.any(1).sum()),improved_changed_points=int(np.sum(changed&(pe<be))),worsened_changed_points=int(np.sum(changed&(pe>be))),equal_changed_points=int(np.sum(changed&(pe==be))),runtime_seconds=time.perf_counter()-start,gradient_model_fits=0,neural_checkpoint_inferences=0,no_GPU=True,no_test_data=True,no_F_split_or_fit=True,no_final_fit_or_submission=True,pressure_predictor_is_input_only=True,runtime=dict(python=platform.python_version(),numpy=np.__version__),artifact_sha256={str(p.relative_to(o)):c33_sha(p) for p in sorted(o.iterdir()) if p.is_file()})
        c33_json(o/'summary.json',result);return result
